# PPO Fine-Tuning on Frozen BC Encoders (A0 vs A1)

Takes the frozen CNN encoders from two behaviour-cloning arms (A0 = pure BC,
A1 = action-conditioned predictive BC) and trains a small PPO actor-critic
head on top of each, in MetaDrive. Goal: check whether the A1 encoder gives
PPO a better starting representation — i.e. does it converge faster than A0?

**Run top to bottom on a fresh Colab runtime with a GPU (Runtime → Change
runtime type → GPU).** Cell 3 is a hard gate — it must print
`HEADLESS_METADRIVE_TEST_PASSED` before you continue; if it fails, stop and
debug there rather than running later cells.

Total budget is 2,000,000 PPO steps per arm — this will very likely span
multiple Colab sessions. Cells 6 and 7 pass `--resume`, so re-running the
training cell after a disconnect picks up from the last saved checkpoint on
Drive instead of restarting.


## Cell 1 — Install Miniconda, create a Python 3.11 env, install MetaDrive

In [ ]:
%%bash
set -e

if [ ! -d /usr/local/miniconda ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /usr/local/miniconda
fi
export PATH="/usr/local/miniconda/bin:$PATH"

# Colab's default Python (3.13) is too new for MetaDrive — build an
# isolated 3.11 env for everything MetaDrive/torch related.
if ! conda env list | grep -q "^py311 "; then
    conda create -y -n py311 python=3.11 -q
fi

conda run -n py311 pip install -q --upgrade pip

# system libs needed for (virtual-display) OpenGL + Panda3D
apt-get update -qq
apt-get install -y -qq xvfb libgl1-mesa-glx libgl1-mesa-dev freeglut3-dev mesa-utils > /dev/null

conda run -n py311 pip install -q \
    metadrive-simulator==0.4.3 \
    torch torchvision \
    numpy pandas opencv-python-headless matplotlib

echo "py311 env ready:"
conda run -n py311 python --version
conda run -n py311 python -c "import metadrive; print('metadrive', metadrive.__version__)"


## Cell 2 — Mount Drive, clone the repo, verify the frozen encoder checkpoints exist

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_URL  = "https://github.com/ShubhamWaghmare11/vision-nav-predictive.git"
REPO_ROOT = "/content/vision-nav-predictive"

if not os.path.isdir(REPO_ROOT):
    !git clone -q {REPO_URL} {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull -q

ENCODER_DIR = "/content/drive/MyDrive/vision-nav/rl_encoders"
A0_ENCODER  = f"{ENCODER_DIR}/a0_500k_s0_final.pt"
A1_ENCODER  = f"{ENCODER_DIR}/a1_500k_s0_final.pt"
RL_RUNS_DIR = "/content/drive/MyDrive/vision-nav/rl_runs"

for path in (A0_ENCODER, A1_ENCODER):
    assert os.path.isfile(path), (
        f"MISSING: {path}\n"
        f"Train/export this BC checkpoint to Drive before running this notebook."
    )
    print(f"found: {path}  ({os.path.getsize(path) / 1e6:.1f} MB)")

os.makedirs(RL_RUNS_DIR, exist_ok=True)
os.makedirs(f"{RL_RUNS_DIR}/a0", exist_ok=True)
os.makedirs(f"{RL_RUNS_DIR}/a1", exist_ok=True)
print("\nDrive + repo ready.")


## Cell 3 — Virtual display + headless MetaDrive smoke test (hard gate)

MetaDrive initialises Panda3D even for offscreen/image-observation use, and
in a headless container that init can crash with a `simplepbr` PBR shader
error because there's no real display for Panda3D's default hardware GL
pipeline to attach to. The fix here is two-part:

1. Start an `Xvfb` virtual X display so Panda3D's GLX path has *something*
   to attach to.
2. Before `metadrive` (and therefore `panda3d.core`) is imported anywhere,
   call `loadPrcFileData(...)` to force Panda3D into an offscreen window
   type and give it an explicit display-pipe fallback chain: try hardware
   `pandagl` first (works through Xvfb when a real GPU is present, as on a
   Colab GPU runtime), falling back to the pure-software `p3tinydisplay`
   pipe if hardware GL isn't available.

This cell does not just start things — it actually calls `env.reset()` and
`env.step()` in a subprocess and asserts on a pass marker in its output. If
this cell raises, **stop here and debug** rather than running the training
cells against a broken renderer.


In [ ]:
import os, subprocess, time

# 1) virtual X display for Panda3D's GLX/EGL init
subprocess.Popen(["Xvfb", ":1", "-screen", "0", "1024x768x24"])
time.sleep(2)
os.environ["DISPLAY"] = ":1"
print("Xvfb started on DISPLAY=:1")


In [ ]:
%%writefile /content/test_headless.py
import os
os.environ.setdefault("DISPLAY", ":1")

# Must run BEFORE metadrive (and therefore panda3d.core) is imported
# anywhere else, or MetaDrive's default GL init runs first and the
# simplepbr shader crash happens before these directives take effect.
from panda3d.core import loadPrcFileData
loadPrcFileData("", "window-type offscreen")
loadPrcFileData("", "load-display pandagl")       # try hardware GL first
loadPrcFileData("", "aux-display pandagl")
loadPrcFileData("", "aux-display p3tinydisplay")  # software fallback
loadPrcFileData("", "framebuffer-software 1")
loadPrcFileData("", "sync-video 0")
loadPrcFileData("", "textures-power-2 none")

import sys
sys.path.insert(0, "/content/vision-nav-predictive")

from metadrive.envs.metadrive_env import MetaDriveEnv
from src.env.metadrive_wrapper import TRAIN_CFG

print("Creating MetaDriveEnv (headless)...")
env = MetaDriveEnv(dict(TRAIN_CFG, num_scenarios=5, start_seed=1000))

obs, info = env.reset(seed=1000)
print("reset() OK — image shape:", obs["image"].shape)

for i in range(10):
    obs, reward, term, trunc, info = env.step([0.0, 0.0])
print("step() x10 OK")

env.close()
print("HEADLESS_METADRIVE_TEST_PASSED")


In [ ]:
import os, subprocess

result = subprocess.run(
    ["/usr/local/miniconda/bin/conda", "run", "-n", "py311", "python", "/content/test_headless.py"],
    env={**os.environ, "DISPLAY": ":1"},
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)

assert "HEADLESS_METADRIVE_TEST_PASSED" in result.stdout, (
    "MetaDrive headless test FAILED — do NOT proceed to the training cells.\n"
    "Check the traceback above. If pandagl fails even through Xvfb, try "
    "commenting out the 'aux-display pandagl' lines in /content/test_headless.py "
    "so only the p3tinydisplay software fallback is used, then re-run this cell."
)
print("\nMetaDrive headless env confirmed working — safe to proceed.")


## Cell 4 — Copy frozen encoders locally, load + verify both

In [ ]:
import os, shutil

os.makedirs("/content/encoders", exist_ok=True)
shutil.copy(A0_ENCODER, "/content/encoders/a0_500k_s0_final.pt")
shutil.copy(A1_ENCODER, "/content/encoders/a1_500k_s0_final.pt")
print("copied encoders to /content/encoders/")


In [ ]:
%%writefile /content/verify_encoders.py
import sys, torch
sys.path.insert(0, "/content/vision-nav-predictive")
from src.models.policy import VisualPolicy

for name, path in [
    ("a0", "/content/encoders/a0_500k_s0_final.pt"),
    ("a1", "/content/encoders/a1_500k_s0_final.pt"),
]:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    cfg = ckpt["config"]
    policy = VisualPolicy(
        chunk_len=cfg.get("chunk_len", 8), K=cfg.get("K", 8),
        d_model=cfg.get("d_model", 256), n_layers=cfg.get("n_layers", 4),
        n_heads=cfg.get("n_heads", 4), dropout=cfg.get("dropout", 0.1),
    )
    policy.load_state_dict(ckpt["policy"])
    encoder = policy.encoder
    n_params = sum(p.numel() for p in encoder.parameters())

    x = torch.randint(0, 255, (2, 3, 84, 84), dtype=torch.uint8).float() / 255.0
    with torch.no_grad():
        z = encoder(x)

    d_model = cfg.get("d_model", 256)
    assert z.shape == (2, d_model), f"unexpected encoder output shape for {name}: {z.shape}"

    print(f"{name}: arm={cfg.get('arm')!r} trained_step={ckpt.get('step')} "
          f"encoder_params={n_params:,} output_shape={tuple(z.shape)}")

print("VERIFY_ENCODERS_PASSED")


In [ ]:
import subprocess

result = subprocess.run(
    ["/usr/local/miniconda/bin/conda", "run", "-n", "py311", "python", "/content/verify_encoders.py"],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
assert "VERIFY_ENCODERS_PASSED" in result.stdout, "Encoder verification failed — see traceback above."


## Cell 5 — PPO training script

Writes the full PPO-with-GAE training loop to `/content/ppo_train.py`. Key
design notes:

- **Frozen encoder**: loads the full BC `VisualPolicy`, keeps only
  `policy.encoder` (the CNN), and freezes it (`requires_grad_(False)`,
  `.eval()`). The BC encoder produces one latent per frame — since the PPO
  head is specified to take a single `(B, 256)` vector, the `K=8` per-frame
  latents are mean-pooled into one vector per step.
- **PPO head**: a small actor-critic (`Linear(256,128) → Tanh → Linear(128,2)`
  for both actor and critic, per the spec) trained with clipped-surrogate
  PPO + GAE. Only the head's parameters are ever passed to the optimizer.
- **Reward**: `Δroute_completion × 10 + 0.01` alive bonus per step,
  `-1.0` on `out_of_road`, `+5.0` on `arrive_dest`.
- **Train seeds**: sampled uniformly from `[1000, 1400)` at the start of
  every episode — same range used for BC training.
- **Resilience**: `--resume` reloads the latest `ppo_ckpt_*.pt` + optimizer
  state + step count from `--out_dir` if one exists, so a 2M-step run can
  safely span multiple Colab sessions.


In [ ]:
%%writefile /content/ppo_train.py
"""
PPO training on top of a frozen BC encoder (A0 or A1) in MetaDrive.

Run:
    python ppo_train.py --encoder_path <ckpt.pt> --arm_name a0 --out_dir <dir>
"""
import os
os.environ.setdefault("DISPLAY", ":1")

# Headless Panda3D setup — must happen before metadrive/panda3d.core is
# imported anywhere else, or MetaDrive's default GL init runs first and the
# simplepbr shader crash happens before these directives take effect.
# Guarded by an env var so this can be smoke-tested on a machine with a
# real display (Colab always has METADRIVE_HEADLESS_FIX=1, the default).
if os.environ.get("METADRIVE_HEADLESS_FIX", "1") == "1":
    from panda3d.core import loadPrcFileData
    loadPrcFileData("", "window-type offscreen")
    loadPrcFileData("", "load-display pandagl")
    loadPrcFileData("", "aux-display pandagl")
    loadPrcFileData("", "aux-display p3tinydisplay")
    loadPrcFileData("", "framebuffer-software 1")
    loadPrcFileData("", "sync-video 0")
    loadPrcFileData("", "textures-power-2 none")

import sys, argparse, json, time, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

REPO_ROOT = os.environ.get("VISION_NAV_REPO", "/content/vision-nav-predictive")
sys.path.insert(0, REPO_ROOT)

from metadrive.envs.metadrive_env import MetaDriveEnv
from src.env.metadrive_wrapper import TRAIN_CFG, build_student_obs, reset_command_state
from src.models.policy import VisualPolicy


# ── frozen encoder wrapper ──────────────────────────────────────────────
class FrozenEncoder(nn.Module):
    """
    Wraps a frozen VisualPolicy CNN encoder. The BC encoder produces one
    latent per frame (B, K, d_model); we mean-pool over the K-frame history
    to get the single (B, d_model) vector the PPO head consumes.
    """
    def __init__(self, encoder, d_model=256):
        super().__init__()
        self.encoder = encoder
        self.d_model = d_model
        for p in self.encoder.parameters():
            p.requires_grad_(False)
        self.encoder.eval()

    @torch.no_grad()
    def forward(self, frames):
        # frames: (B, K, 3, 84, 84) uint8
        B, K, C, H, W = frames.shape
        flat = frames.reshape(B * K, C, H, W).float() / 255.0
        z = self.encoder(flat).view(B, K, self.d_model)
        return z.mean(dim=1)


def load_frozen_encoder(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ckpt["config"]
    policy = VisualPolicy(
        chunk_len=cfg.get("chunk_len", 8),
        K=cfg.get("K", 8),
        d_model=cfg.get("d_model", 256),
        n_layers=cfg.get("n_layers", 4),
        n_heads=cfg.get("n_heads", 4),
        dropout=cfg.get("dropout", 0.1),
    )
    policy.load_state_dict(ckpt["policy"])
    frozen = FrozenEncoder(policy.encoder, d_model=cfg.get("d_model", 256)).to(device)
    return frozen, cfg.get("K", 8)


# ── PPO actor-critic head ──────────────────────────────────────────────
class ActorCriticHead(nn.Module):
    def __init__(self, d_model=256, hidden=128, act_dim=2):
        super().__init__()
        self.actor_body = nn.Sequential(nn.Linear(d_model, hidden), nn.Tanh())
        self.actor_mean = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim) - 0.5)

        self.critic = nn.Sequential(
            nn.Linear(d_model, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, z):
        mean = torch.tanh(self.actor_mean(self.actor_body(z)))
        std = self.log_std.exp().expand_as(mean)
        value = self.critic(z).squeeze(-1)
        return mean, std, value

    def act(self, z):
        mean, std, value = self.forward(z)
        dist = Normal(mean, std)
        raw_action = dist.rsample()
        logp = dist.log_prob(raw_action).sum(-1)
        action_clipped = raw_action.clamp(-1.0, 1.0)
        return action_clipped, raw_action, logp, value

    def evaluate(self, z, raw_action):
        mean, std, value = self.forward(z)
        dist = Normal(mean, std)
        logp = dist.log_prob(raw_action).sum(-1)
        entropy = dist.entropy().sum(-1)
        return logp, entropy, value


# ── rollout buffer ───────────────────────────────────────────────────────
class RolloutBuffer:
    def __init__(self):
        self.frames, self.raw_actions, self.logps, self.values = [], [], [], []
        self.rewards, self.dones = [], []

    def add(self, frames, raw_action, logp, value, reward, done):
        self.frames.append(frames)
        self.raw_actions.append(raw_action)
        self.logps.append(logp)
        self.values.append(value)
        self.rewards.append(reward)
        self.dones.append(done)


def compute_gae(rewards, values, dones, last_value, gamma, lam):
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    values_ext = values + [last_value]
    for t in reversed(range(T)):
        mask = 1.0 - dones[t]
        delta = rewards[t] + gamma * values_ext[t + 1] * mask - values_ext[t]
        gae = delta + gamma * lam * mask * gae
        advantages[t] = gae
    returns = advantages + np.array(values, dtype=np.float32)
    return advantages, returns


# ── env / obs helpers ────────────────────────────────────────────────────
def make_env(seed_lo, seed_hi):
    cfg = dict(TRAIN_CFG)
    cfg["num_scenarios"] = seed_hi - seed_lo
    cfg["start_seed"] = seed_lo
    return MetaDriveEnv(cfg)


def reset_episode(env, K, seed):
    raw_obs, _ = env.reset(seed=seed)
    reset_command_state(env)
    student = build_student_obs(env, raw_obs)
    frame = torch.from_numpy(student.image.transpose(2, 0, 1).copy())
    frame_buf = [frame] * K
    return student, frame_buf


def stack_obs(frame_buf):
    return torch.stack(frame_buf, dim=0).unsqueeze(0)  # (1, K, 3, 84, 84) uint8


def reward_fn(prev_completion, info):
    completion = float(info.get("route_completion", prev_completion))
    delta = completion - prev_completion
    r = delta * 10.0 + 0.01
    if info.get("out_of_road", False):
        r -= 1.0
    if info.get("arrive_dest", False):
        r += 5.0
    return r, completion


# ── main PPO loop ─────────────────────────────────────────────────────────
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--encoder_path", required=True)
    p.add_argument("--arm_name", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--total_steps", type=int, default=2_000_000)
    p.add_argument("--rollout_len", type=int, default=512)
    p.add_argument("--ppo_epochs", type=int, default=4)
    p.add_argument("--minibatch", type=int, default=64)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--gae_lambda", type=float, default=0.95)
    p.add_argument("--clip_eps", type=float, default=0.2)
    p.add_argument("--seed_lo", type=int, default=1000)
    p.add_argument("--seed_hi", type=int, default=1400)
    p.add_argument("--ckpt_every", type=int, default=100)
    p.add_argument("--resume", action="store_true")
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}", flush=True)

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.json"

    encoder, K = load_frozen_encoder(args.encoder_path, device)
    head = ActorCriticHead(d_model=256).to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=args.lr)

    global_step, update = 0, 0
    metrics = []

    ckpts = sorted(out_dir.glob("ppo_ckpt_*.pt"))
    if args.resume and ckpts:
        latest = ckpts[-1]
        ckpt = torch.load(latest, map_location=device, weights_only=False)
        head.load_state_dict(ckpt["head"])
        optimizer.load_state_dict(ckpt["optimizer"])
        global_step = ckpt["global_step"]
        update = ckpt["update"]
        if metrics_path.exists():
            metrics = json.loads(metrics_path.read_text())
        print(f"Resumed from {latest.name} at step {global_step}", flush=True)

    env = make_env(args.seed_lo, args.seed_hi)

    episode_count = 0
    ep_reward, ep_len, ep_completion = 0.0, 0, 0.0
    recent_rewards, recent_lens, recent_completions = [], [], []

    seed = random.randint(args.seed_lo, args.seed_hi - 1)
    student, frame_buf = reset_episode(env, K, seed)
    prev_completion = 0.0

    t_log = time.time()
    steps_since_log = 0

    try:
        while global_step < args.total_steps:
            buf = RolloutBuffer()

            for _ in range(args.rollout_len):
                frames = stack_obs(frame_buf).to(device)
                with torch.no_grad():
                    z = encoder(frames)
                    action_clipped, raw_action, logp, value = head.act(z)

                action_np = action_clipped[0].cpu().numpy()
                raw_obs, _, term, trunc, info = env.step(action_np.tolist())
                reward, completion = reward_fn(prev_completion, info)
                prev_completion = completion
                done = term or trunc

                buf.add(frames.cpu(), raw_action.cpu(), logp.cpu(), value.item(), reward, float(done))

                ep_reward += reward
                ep_len += 1
                ep_completion = completion
                global_step += 1
                steps_since_log += 1

                student = build_student_obs(env, raw_obs)
                frame = torch.from_numpy(student.image.transpose(2, 0, 1).copy())
                frame_buf.append(frame)
                frame_buf.pop(0)

                if done:
                    episode_count += 1
                    recent_rewards.append(ep_reward)
                    recent_lens.append(ep_len)
                    recent_completions.append(ep_completion)

                    if episode_count % 10 == 0:
                        sps = steps_since_log / max(time.time() - t_log, 1e-6)
                        print(
                            f"step {global_step:>9d} | ep {episode_count:>5d} | "
                            f"reward {np.mean(recent_rewards[-10:]):.3f} | "
                            f"len {np.mean(recent_lens[-10:]):.1f} | "
                            f"completion {np.mean(recent_completions[-10:]):.3f} | "
                            f"sps {sps:.1f}",
                            flush=True,
                        )
                        t_log = time.time()
                        steps_since_log = 0

                    seed = random.randint(args.seed_lo, args.seed_hi - 1)
                    student, frame_buf = reset_episode(env, K, seed)
                    prev_completion = 0.0
                    ep_reward, ep_len, ep_completion = 0.0, 0, 0.0

                if global_step >= args.total_steps:
                    break

            frames = stack_obs(frame_buf).to(device)
            with torch.no_grad():
                z = encoder(frames)
                _, _, last_value = head.forward(z)
            last_value = last_value.item()

            advantages, returns = compute_gae(
                buf.rewards, buf.values, buf.dones, last_value, args.gamma, args.gae_lambda
            )
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

            b_frames = torch.cat(buf.frames, dim=0)
            b_actions = torch.cat(buf.raw_actions, dim=0)
            b_logp = torch.cat(buf.logps, dim=0)
            b_adv = torch.from_numpy(advantages)
            b_ret = torch.from_numpy(returns)

            n = b_frames.shape[0]
            idx = np.arange(n)

            for _ in range(args.ppo_epochs):
                np.random.shuffle(idx)
                for start in range(0, n, args.minibatch):
                    mb = idx[start:start + args.minibatch]
                    mb_frames = b_frames[mb].to(device)
                    mb_actions = b_actions[mb].to(device)
                    mb_logp_old = b_logp[mb].to(device)
                    mb_adv = b_adv[mb].to(device)
                    mb_ret = b_ret[mb].to(device)

                    with torch.no_grad():
                        z = encoder(mb_frames)
                    logp, entropy, value = head.evaluate(z, mb_actions)

                    ratio = (logp - mb_logp_old).exp()
                    surr1 = ratio * mb_adv
                    surr2 = ratio.clamp(1 - args.clip_eps, 1 + args.clip_eps) * mb_adv
                    policy_loss = -torch.min(surr1, surr2).mean()
                    value_loss = F.mse_loss(value, mb_ret)
                    entropy_loss = -entropy.mean()

                    loss = policy_loss + 0.5 * value_loss + 0.01 * entropy_loss

                    optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(head.parameters(), 0.5)
                    optimizer.step()

            update += 1

            if update % args.ckpt_every == 0:
                ckpt_path = out_dir / f"ppo_ckpt_{update:06d}.pt"
                torch.save({
                    "update": update, "global_step": global_step,
                    "head": head.state_dict(), "optimizer": optimizer.state_dict(),
                    "arm_name": args.arm_name,
                }, ckpt_path)

                metrics.append({
                    "update": update,
                    "global_step": global_step,
                    "episode_count": episode_count,
                    "mean_reward_last10": float(np.mean(recent_rewards[-10:])) if recent_rewards else 0.0,
                    "mean_completion_last10": float(np.mean(recent_completions[-10:])) if recent_completions else 0.0,
                    "time": time.time(),
                })
                metrics_path.write_text(json.dumps(metrics, indent=2))
                print(f"  -> checkpoint + metrics saved at update {update} (step {global_step})", flush=True)

        final_path = out_dir / "ppo_ckpt_final.pt"
        torch.save({
            "update": update, "global_step": global_step,
            "head": head.state_dict(), "optimizer": optimizer.state_dict(),
            "arm_name": args.arm_name,
        }, final_path)
        metrics_path.write_text(json.dumps(metrics, indent=2))
        print("PPO training complete.", flush=True)
    finally:
        env.close()


if __name__ == "__main__":
    main()


## Cell 6 — Train PPO on the frozen A0 encoder

In [ ]:
!/usr/local/miniconda/bin/conda run -n py311 env DISPLAY=:1 VISION_NAV_REPO=/content/vision-nav-predictive \
  python -u /content/ppo_train.py \
    --encoder_path /content/encoders/a0_500k_s0_final.pt \
    --arm_name a0 \
    --out_dir /content/drive/MyDrive/vision-nav/rl_runs/a0 \
    --total_steps 2000000 \
    --rollout_len 512 \
    --ppo_epochs 4 \
    --minibatch 64 \
    --lr 3e-4 \
    --gamma 0.99 \
    --gae_lambda 0.95 \
    --clip_eps 0.2 \
    --seed_lo 1000 --seed_hi 1400 \
    --ckpt_every 100 \
    --resume


## Cell 7 — Train PPO on the frozen A1 encoder

In [ ]:
!/usr/local/miniconda/bin/conda run -n py311 env DISPLAY=:1 VISION_NAV_REPO=/content/vision-nav-predictive \
  python -u /content/ppo_train.py \
    --encoder_path /content/encoders/a1_500k_s0_final.pt \
    --arm_name a1 \
    --out_dir /content/drive/MyDrive/vision-nav/rl_runs/a1 \
    --total_steps 2000000 \
    --rollout_len 512 \
    --ppo_epochs 4 \
    --minibatch 64 \
    --lr 3e-4 \
    --gamma 0.99 \
    --gae_lambda 0.95 \
    --clip_eps 0.2 \
    --seed_lo 1000 --seed_hi 1400 \
    --ckpt_every 100 \
    --resume


## Cell 8 — Plot learning curves: A0 vs A1

In [ ]:
import json
import matplotlib.pyplot as plt

RL_RUNS_DIR = "/content/drive/MyDrive/vision-nav/rl_runs"

def load_metrics(arm):
    with open(f"{RL_RUNS_DIR}/{arm}/metrics.json") as f:
        return json.load(f)

a0_metrics = load_metrics("a0")
a1_metrics = load_metrics("a1")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metrics, label, color in [
    (a0_metrics, "A0 (pure BC)",        "tab:blue"),
    (a1_metrics, "A1 (predictive BC)",  "tab:orange"),
]:
    steps       = [m["global_step"] for m in metrics]
    rewards     = [m["mean_reward_last10"] for m in metrics]
    completions = [m["mean_completion_last10"] for m in metrics]
    axes[0].plot(steps, rewards, label=label, color=color)
    axes[1].plot(steps, completions, label=label, color=color)

axes[0].set_xlabel("PPO training steps")
axes[0].set_ylabel("Mean episode reward (last 10 eps)")
axes[0].set_title("Episode reward vs. training steps")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_xlabel("PPO training steps")
axes[1].set_ylabel("Mean route completion (last 10 eps)")
axes[1].set_title("Route completion vs. training steps")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out_png = f"{RL_RUNS_DIR}/a0_vs_a1_learning_curves.png"
plt.savefig(out_png, dpi=150)
plt.show()
print(f"saved: {out_png}")
